# boursorama_peapme

1. Lecture du listing Boursorama PEA-PME par lettre (A..Z + 0)
2. Extraction des infos marché depuis `data-ist-init`
3. Suivi du lien détail de chaque société
4. Extraction de l’ISIN dans la fiche valeur
5. Export CSV final prêt pour le mapping yfinance


In [1]:
# Imports
import csv
import html
import json
import re
import time
from typing import Any, Dict, Optional, List
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# Configuration générale

BASE_URL = "https://www.boursorama.com/bourse/actions/cotations/"
MARKET = "PEAPME"
LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + ["0"]

# Délai recommandé entre requêtes pour rester poli vis-à-vis du site
DELAY_SECONDS = 0.3

# CSV de sortie
OUTPUT_CSV = "boursorama_peapme_final.csv"

# En-têtes HTTP : User-Agent obligatoire
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
}

In [3]:
# Session HTTP avec retries


def make_session() -> requests.Session:
    """
    Crée une session requests réutilisable.
    On ajoute des retries sur les codes transitoires pour rendre le script
    plus robuste sur une grande série de requêtes.
    """
    session = requests.Session()
    session.headers.update(HEADERS)

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = make_session()


def fetch_html(url: str, params: Optional[dict] = None) -> str:
    """
    Fait un GET et renvoie le HTML brut.
    """
    response = SESSION.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.text

In [4]:
# Helpers de parsing JSON / recherche de valeurs imbriquées


def extract_json_like(text: str) -> Any:
    """
    Tente de parser une chaîne potentiellement JSON encodée / échappée.
    Certaines valeurs HTML peuvent contenir des entités ou du JSON inline.
    """
    if not text:
        return {}

    # Décodage des entités HTML éventuelles
    text = html.unescape(text).strip()

    # 1) Essai direct
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2) Si du texte parasite entoure le JSON, on récupère le premier bloc {...}
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return {}

    return {}


def find_first_value(obj: Any, keys: List[str]) -> Optional[Any]:
    """
    Parcourt récursivement un objet dict/list et renvoie la première valeur
    non vide trouvée pour une des clés demandées.
    """
    if isinstance(obj, dict):
        for key in keys:
            if key in obj and obj[key] not in (None, "", [], {}):
                return obj[key]
        for value in obj.values():
            found = find_first_value(value, keys)
            if found is not None:
                return found

    elif isinstance(obj, list):
        for item in obj:
            found = find_first_value(item, keys)
            if found is not None:
                return found

    return None

In [5]:
# Étape 1 : parsing du listing


def parse_listing_row(tr) -> Optional[Dict[str, Any]]:
    """
    Parse une ligne <tr> du listing.
    On récupère :
    - ticker_bourso depuis data-ist
    - les données de marché depuis data-ist-init
    - le lien détail pour l'étape 2
    """
    ticker_bourso = tr.get("data-ist")
    data_ist_init_raw = tr.get("data-ist-init", "")

    # Lien détail : il vaut mieux le suivre directement que le reconstruire
    link = tr.select_one("a[href*='/cours/']")
    detail_href = link.get("href") if link and link.has_attr("href") else None
    detail_url = urljoin(BASE_URL, detail_href) if detail_href else None

    # Le nom visible est souvent dans le lien
    name = link.get_text(" ", strip=True) if link else None
    if not name:
        # Fallback si la structure change légèrement
        text = tr.get_text(" ", strip=True)
        name = text or None

    # Parsing du JSON inline des marchés
    payload = extract_json_like(data_ist_init_raw)

    # Les clés exactes peuvent varier selon le site / l'objet embarqué.
    # On cherche donc les noms les plus probables.
    last = find_first_value(payload, ["last", "lastPrice", "price", "value"])
    variation = find_first_value(payload, ["variation", "change", "pctChange", "variationPercent"])
    high = find_first_value(payload, ["high", "dayHigh", "highPrice"])
    low = find_first_value(payload, ["low", "dayLow", "lowPrice"])
    volume = find_first_value(payload, ["volume", "tradedVolume", "turnover"])

    # Si la ligne est vide ou inutilisable, on l'ignore
    if not any([ticker_bourso, name, detail_url]):
        return None

    return {
        "name": name,
        "ticker_bourso": ticker_bourso,
        "detail_url": detail_url,
        "last": last,
        "variation": variation,
        "high": high,
        "low": low,
        "volume": volume,
        # Garde utile pour debug si vous voulez inspecter la structure brute
        "raw_data_ist_init": data_ist_init_raw,
    }


def scrape_listing_for_letter(letter: str) -> List[Dict[str, Any]]:
    """
    Récupère toutes les sociétés affichées pour une lettre donnée.
    Pas de pagination ici : on prend toutes les lignes du tableau statique.
    """
    params = {
        "quotation_az_filter[market]": MARKET,
        "quotation_az_filter[letter]": letter,
    }

    html_text = fetch_html(BASE_URL, params=params)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    for tr in soup.select("tr[data-ist-init]"):
        item = parse_listing_row(tr)
        if item:
            rows.append(item)

    return rows

In [6]:
# Étape 2 : enrichissement ISIN depuis la page détail

ISIN_RE = re.compile(r"\b[A-Z]{2}[A-Z0-9]{10}\b")


def extract_isin(detail_html: str) -> Optional[str]:
    """
    Extrait l'ISIN depuis la fiche détail.
    Cible principale : h2.c-faceplate__isin
    Fallback : regex sur le texte complet de la page
    """
    soup = BeautifulSoup(detail_html, "html.parser")

    # Cible principale
    h2 = soup.select_one("h2.c-faceplate__isin")
    if h2:
        text = h2.get_text(" ", strip=True)
        match = ISIN_RE.search(text)
        if match:
            return match.group(0)

    # Fallback si le sélecteur change légèrement
    page_text = soup.get_text(" ", strip=True)
    match = ISIN_RE.search(page_text)
    if match:
        return match.group(0)

    return None


def enrich_with_isin(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    Télécharge la page détail et ajoute l'ISIN à la ligne courante.
    """
    detail_url = row.get("detail_url")
    if not detail_url:
        row["isin"] = ""
        return row

    detail_html = fetch_html(detail_url)
    row["isin"] = extract_isin(detail_html) or ""
    return row

In [7]:
# Programme principal


def main():
    """
    Pipeline complet :
    1) scrape listing sur 27 lettres
    2) dédoublonnage léger
    3) enrichissement ISIN
    4) export CSV final
    """
    all_rows: List[Dict[str, Any]] = []
    seen = set()

    # Étape 1 : listing
    for letter in LETTERS:
        print(f"[LISTING] Lettre {letter}...")
        rows = scrape_listing_for_letter(letter)

        for row in rows:
            # Clé de dédoublonnage :
            # on privilégie ticker_bourso, sinon le couple (nom, URL détail)
            key = row.get("ticker_bourso") or (row.get("name"), row.get("detail_url"))
            if key in seen:
                continue
            seen.add(key)
            all_rows.append(row)

        time.sleep(DELAY_SECONDS)

    print(f"[LISTING] Total lignes uniques : {len(all_rows)}")

    # Étape 2 : enrichissement ISIN
    for idx, row in enumerate(all_rows, start=1):
        name = row.get("name", "")
        print(f"[ISIN] {idx}/{len(all_rows)} - {name}")

        try:
            enrich_with_isin(row)
        except Exception as exc:
            # On ne bloque pas tout le run si une fiche pose problème
            row["isin"] = ""
            row["isin_error"] = str(exc)

        time.sleep(DELAY_SECONDS)

    # Export final
    fieldnames = [
        "name",
        "ticker_bourso",
        "isin",
        "detail_url",
        "last",
        "variation",
        "high",
        "low",
        "volume",
        "raw_data_ist_init",
    ]

    # On ajoute la colonne d'erreur seulement si elle existe quelque part
    if any("isin_error" in r for r in all_rows):
        fieldnames.append("isin_error")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"[OK] CSV final écrit dans : {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[LISTING] Lettre A...
[LISTING] Lettre B...
[LISTING] Lettre C...
[LISTING] Lettre D...
[LISTING] Lettre E...
[LISTING] Lettre F...
[LISTING] Lettre G...
[LISTING] Lettre H...
[LISTING] Lettre I...
[LISTING] Lettre J...
[LISTING] Lettre K...
[LISTING] Lettre L...
[LISTING] Lettre M...
[LISTING] Lettre N...
[LISTING] Lettre O...
[LISTING] Lettre P...
[LISTING] Lettre Q...
[LISTING] Lettre R...
[LISTING] Lettre S...
[LISTING] Lettre T...
[LISTING] Lettre U...
[LISTING] Lettre V...
[LISTING] Lettre W...
[LISTING] Lettre X...
[LISTING] Lettre Y...
[LISTING] Lettre Z...
[LISTING] Lettre 0...
[LISTING] Total lignes uniques : 570
[ISIN] 1/570 - AB SCIENCE
[ISIN] 2/570 - ABC ARBITRAGE
[ISIN] 3/570 - ABEO
[ISIN] 4/570 - ABIONYX PHARMA
[ISIN] 5/570 - ABIVAX
[ISIN] 6/570 - ABL DIAGNOSTICS
[ISIN] 7/570 - ACHETER-LOUER.FR
[ISIN] 8/570 - ACTEOS (EX DATATRONIC)
[ISIN] 9/570 - ACTIA GROUP
[ISIN] 10/570 - ACTICOR BIOTECH
[ISIN] 11/570 - ACTIVIUM GROUP
[ISIN] 12/570 - ADEUNIS
[ISIN] 13/570 - ADOCIA
[ISI

KeyboardInterrupt: 